In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

# Khởi tạo Spark Session
spark = SparkSession.builder.appName("Yellow").getOrCreate()

print("Spark Session đã được khởi tạo thành công!")


Spark Session đã được khởi tạo thành công!


In [2]:
yellow = spark.read.parquet("hdfs://master:9000/data/parquet/yellow_tripdata_2024-*.parquet")

In [3]:
yellow.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 00:57:55|  2024-01-01 01:17:43|              1|         1.72|         1|                 N|         186|          79|           2|       17.7|  1.0|    0.5|       0.

In [4]:
print(yellow.count())


41169720


In [5]:
#IMPORT thư viện
from pyspark.sql.functions import col, count, avg, hour, dayofweek,unix_timestamp, to_timestamp,month, dayofmonth, to_date


In [6]:
yellow_filtered = yellow.filter(
    (col("tpep_pickup_datetime").isNotNull()) &
    (col("tpep_dropoff_datetime").isNotNull()) &
    (col("PULocationID").isNotNull()) &
    (col("DOLocationID").isNotNull())
)

# Chuyển đổi sang timestamp để tính toán thời gian chuyến đi
yellow_filtered = yellow_filtered.withColumn("pickup_ts", to_timestamp(col("tpep_pickup_datetime"))) \
                         .withColumn("dropoff_ts", to_timestamp(col("tpep_dropoff_datetime")))

# 2. Lọc các giá trị PULocationID và DOLocationID hợp lệ
# Giả định ID từ 1 đến 263 là hợp lệ cho các khu vực taxi của NYC
yellow_filtered = yellow_filtered.filter(
    (col("PULocationID") > 0) & (col("PULocationID") <= 263) &
    (col("DOLocationID") > 0) & (col("DOLocationID") <= 263)
)

# 3. Lọc bỏ các chuyến đi có thời gian và quãng đường phi lý
# Tính toán lại trip_time dựa trên timestamp để đảm bảo chính xác
# (đơn vị giây)
yellow_filtered = yellow_filtered.withColumn(
    "calculated_trip_time_seconds",
    unix_timestamp(col("dropoff_ts")) - unix_timestamp(col("pickup_ts"))
)

# Tiêu chí lọc:
# - Thời gian chuyến đi: ít nhất 1 phút (60 giây), tối đa 5 giờ (18000 giây)
# - Quãng đường chuyến đi: ít nhất 0.1 dặm, tối đa 50 dặm
# - Tốc độ trung bình: giữa 1 mph và 60 mph
#    Tốc độ (mph) = (trip_miles / calculated_trip_time_seconds) * 3600
yellow_filtered =yellow_filtered.withColumn(
    "calculated_average_speed_mph",
    (col("trip_distance") / col("calculated_trip_time_seconds")) * 3600
)

yellow_filtered =yellow_filtered.filter(
    (col("calculated_trip_time_seconds") >= 60) & (col("calculated_trip_time_seconds") <= 18000) & # 1 phút đến 5 giờ
    (col("trip_distance") >= 0.1) & (col("trip_distance") <= 50) & # 0.1 dặm đến 50 dặm
    (col("calculated_average_speed_mph") >= 1) & (col("calculated_average_speed_mph") <= 60) # 1 mph đến 60 mph
)

# 4. Lọc các giá trị tài chính hợp lệ (không âm, không quá lớn phi lý)
# Các cột tài chính cần lọc: base_passenger_fare, tolls, bcf, sales_tax, congestion_surcharge, airport_fee, tips, driver_pay
# Đảm bảo không âm và có giới hạn trên hợp lý (ví dụ: < 1000$ cho hầu hết các khoản)
yellow_filtered = yellow_filtered.filter(
    (col("passenger_count") > 0) &
    (col("extra") >= 0) & (col("tip_amount") >= 0) &
    (col("tolls_amount") >= 0) & (col("total_amount") > 0) &
    (col("congestion_surcharge") >= 0) & (col("congestion_surcharge") < 20) &
    (col("airport_fee") >= 0)
)

print(f"Tổng số dòng sau khi lọc: {yellow_filtered.count()}")
yellow = yellow_filtered

Tổng số dòng sau khi lọc: 35091216


In [7]:
from pyspark.sql.functions import col, when, desc, count, to_date, hour, dayofweek, dayofmonth, month, lag
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, IndexToString
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator  # <-- THÊM CÔNG CỤ TUNING
import pandas as pd

In [8]:
print("\n--- BƯỚC 1: Đang tổng hợp dữ liệu giao dịch thành chuỗi thời gian theo giờ... ---")
# a. Gộp các chuyến đi riêng lẻ thành số lượng chuyến đi theo giờ cho mỗi địa điểm
hourly_demand = yellow.groupBy(
    to_date(col("tpep_pickup_datetime")).alias("date"), 
    hour(col("tpep_pickup_datetime")).alias("hour"),
    col("PULocationID")
).agg(count("*").alias("trip_count"))

# b. Thêm các đặc trưng thời gian cơ bản
model_data = hourly_demand.withColumn("day_of_week", dayofweek(col("date"))) \
                          .withColumn("day_of_month", dayofmonth(col("date"))) \
                          .withColumn("month", month(col("date")))


--- BƯỚC 1: Đang tổng hợp dữ liệu giao dịch thành chuỗi thời gian theo giờ... ---


In [9]:
print("\n--- BƯỚC 2: Đang tạo các đặc trưng nâng cao (Kỳ nghỉ và Dữ liệu quá khứ)... ---")

# a. Thêm đặc trưng "Kỳ nghỉ" (Holiday Period)
holiday_list = [
    ("2024-01-01", "Holiday"), ("2024-01-15", "Holiday"), ("2024-05-27", "Holiday"), 
    ("2024-07-04", "Holiday"), ("2024-09-02", "Holiday"), ("2024-11-28", "Holiday"), 
    ("2024-12-25", "Holiday")
]
holiday_periods_list = []
for holiday_date_str, type in holiday_list:
    holiday_date = pd.to_datetime(holiday_date_str)
    holiday_periods_list.append((str(holiday_date.date()), type))
    holiday_periods_list.append((str((holiday_date - pd.DateOffset(days=1)).date()), "Pre-Holiday"))
    holiday_periods_list.append((str((holiday_date + pd.DateOffset(days=1)).date()), "Post-Holiday"))

holiday_periods_df = spark.createDataFrame(holiday_periods_list, ["date_str", "day_type"]) \
                          .withColumn("date", to_date(col("date_str"))).drop("date_str")

model_data_with_holidays = model_data.join(holiday_periods_df, ["date"], "left") \
                                     .na.fill({"day_type": "Normal_Day"})

# b. Thêm đặc trưng Trễ (Lag Feature) - Nhu cầu của cùng giờ ngày hôm qua
window_spec = Window.partitionBy("PULocationID").orderBy("date", "hour")
data_with_features = model_data_with_holidays.withColumn("lag_24hr", lag("trip_count", 24).over(window_spec)) \
                                             .na.fill(0, subset=["lag_24hr"])



--- BƯỚC 2: Đang tạo các đặc trưng nâng cao (Kỳ nghỉ và Dữ liệu quá khứ)... ---


In [10]:
print("\n--- BƯỚC 3: Đang tạo nhãn phân loại và trọng số để xử lý mất cân bằng... ---")

# a. Xác định các ngưỡng nhu cầu một cách tự động
quantiles = data_with_features.approxQuantile("trip_count", [0.25, 0.5, 0.75, 0.95], 0.01)
data_with_levels = data_with_features.withColumn("demand_level",
    when(col("trip_count") == 0, "None")
    .when(col("trip_count") <= quantiles[0], "Very_Low")
    .when(col("trip_count") <= quantiles[1], "Low")
    .when(col("trip_count") <= quantiles[2], "Medium")
    .when(col("trip_count") <= quantiles[3], "High")
    .otherwise("Very_High")
)

# b. Tính toán trọng số cho mỗi lớp để xử lý mất cân bằng
class_counts = data_with_levels.groupBy("demand_level").count()
total_samples = data_with_levels.count()
num_classes = class_counts.count()
calculate_weight = class_counts.withColumn("weight", total_samples / (num_classes * col("count")))
final_data_for_pipeline = data_with_levels.join(calculate_weight.select("demand_level", "weight"), ["demand_level"], "left")


--- BƯỚC 3: Đang tạo nhãn phân loại và trọng số để xử lý mất cân bằng... ---


In [11]:
print("\n--- BƯỚC 4: Bắt đầu xây dựng, huấn luyện và đánh giá Pipeline... ---")

# a. Định nghĩa các bước trong Pipeline
label_indexer = StringIndexer(inputCol="demand_level", outputCol="label", handleInvalid="keep")
day_type_indexer = StringIndexer(inputCol="day_type", outputCol="day_type_index", handleInvalid="keep")
day_type_encoder = OneHotEncoder(inputCol="day_type_index", outputCol="day_type_vec")
feature_columns = ["PULocationID", "hour", "day_of_week", "day_of_month", "month", "lag_24hr", "day_type_vec"]
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features", handleInvalid="skip")
rf_classifier = RandomForestClassifier(featuresCol="features", labelCol="label", weightCol="weight", seed=42)
pipeline = Pipeline(stages=[label_indexer, day_type_indexer, day_type_encoder, assembler, rf_classifier])

# b. Chia dữ liệu Train/Test
train_data, test_data = final_data_for_pipeline.randomSplit([0.8, 0.2], seed=42)

# c. Huấn luyện Pipeline
pipeline_model = pipeline.fit(train_data)
print("Đã huấn luyện xong Pipeline!")

# d. Thực hiện dự đoán
predictions = pipeline_model.transform(test_data)

# e. Đánh giá mô hình
print("\n--- Bảng kết quả đánh giá mô hình ---")
evaluator_accuracy = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
accuracy = evaluator_accuracy.evaluate(predictions)
f1_score = evaluator_f1.evaluate(predictions)
print(f"Độ chính xác (Accuracy): {accuracy * 100:.2f}%")
print(f"Chỉ số F1-Score (trung bình): {f1_score:.2f}")

# f. Hiển thị kết quả chi tiết
print("\n--- Ma trận Nhầm lẫn (Confusion Matrix) ---")
labels = pipeline_model.stages[0].labels
prediction_converter = IndexToString(inputCol="prediction", outputCol="predicted_level", labels=labels)
final_output = prediction_converter.transform(predictions)
final_output.groupBy("demand_level").pivot("predicted_level", labels).count().na.fill(0).show()


--- BƯỚC 4: Bắt đầu xây dựng, huấn luyện và đánh giá Pipeline... ---
Đã huấn luyện xong Pipeline!

--- Bảng kết quả đánh giá mô hình ---
Độ chính xác (Accuracy): 65.05%
Chỉ số F1-Score (trung bình): 0.63

--- Ma trận Nhầm lẫn (Confusion Matrix) ---
+------------+--------+------+----+-----+---------+
|demand_level|Very_Low|Medium| Low| High|Very_High|
+------------+--------+------+----+-----+---------+
|        High|     139|  3741|  91|25911|     4174|
|         Low|   22901|  5162|9946|  219|       38|
|   Very_High|      25|    56|   8| 1519|     7874|
|      Medium|    3473| 27187|4108| 4731|      244|
|    Very_Low|   40975|  1952|7075|  101|       28|
+------------+--------+------+----+-----+---------+



In [12]:
print("\n--- Cách 2, bước 2: Đang tạo các đặc trưng nâng cao (Kỳ nghỉ và Dữ liệu quá khứ)... ---")

# a. Thêm đặc trưng "Kỳ nghỉ" (Holiday Period)
holiday_list = [
    ("2024-01-01", "Holiday"), ("2024-01-15", "Holiday"), ("2024-05-27", "Holiday"), 
    ("2024-07-04", "Holiday"), ("2024-09-02", "Holiday"), ("2024-11-28", "Holiday"), 
    ("2024-12-25", "Holiday")
]
holiday_periods_list = []
for holiday_date_str, type in holiday_list:
    holiday_date = pd.to_datetime(holiday_date_str)
    holiday_periods_list.append((str(holiday_date.date()), type))
    holiday_periods_list.append((str((holiday_date - pd.DateOffset(days=1)).date()), "Pre-Holiday"))
    holiday_periods_list.append((str((holiday_date + pd.DateOffset(days=1)).date()), "Post-Holiday"))

holiday_periods_df = spark.createDataFrame(holiday_periods_list, ["date_str", "day_type"]) \
                          .withColumn("date", to_date(col("date_str"))).drop("date_str")

model_data_with_holidays = model_data.join(holiday_periods_df, ["date"], "left") \
                                     .na.fill({"day_type": "Normal_Day"})

# b. Thêm đặc trưng Trễ (Lag Feature) - Nhu cầu của cùng giờ ngày hôm qua
window_spec = Window.partitionBy("PULocationID").orderBy("date", "hour")
data_with_features = model_data_with_holidays.withColumn("lag_24hr", lag("trip_count", 24).over(window_spec)) \
                                             .withColumn("lag_168hr", lag("trip_count", 24 * 7).over(window_spec)) \
                                             .na.fill(0, subset=["lag_24hr", "lag_168hr"])


--- Cách 2, bước 2: Đang tạo các đặc trưng nâng cao (Kỳ nghỉ và Dữ liệu quá khứ)... ---


In [13]:
print("\n--- CÁch 2_BƯỚC 3: Đang tạo nhãn phân loại và trọng số để xử lý mất cân bằng... ---")

# a. Xác định các ngưỡng nhu cầu một cách tự động
quantiles = data_with_features.approxQuantile("trip_count", [0.25, 0.5, 0.75, 0.95], 0.01)
data_with_levels = data_with_features.withColumn("demand_level",
    when(col("trip_count") == 0, "None")
    .when(col("trip_count") <= quantiles[0], "Very_Low")
    .when(col("trip_count") <= quantiles[1], "Low")
    .when(col("trip_count") <= quantiles[2], "Medium")
    .when(col("trip_count") <= quantiles[3], "High")
    .otherwise("Very_High")
)

# b. Tính toán trọng số cho mỗi lớp để xử lý mất cân bằng
class_counts = data_with_levels.groupBy("demand_level").count()
total_samples = data_with_levels.count()
num_classes = class_counts.count()
calculate_weight = class_counts.withColumn("weight", total_samples / (num_classes * col("count")))
final_data_for_pipeline = data_with_levels.join(calculate_weight.select("demand_level", "weight"), ["demand_level"], "left")


--- CÁch 2_BƯỚC 3: Đang tạo nhãn phân loại và trọng số để xử lý mất cân bằng... ---


In [14]:
print("\n--- Cách 2_BƯỚC 4: Bắt đầu xây dựng, huấn luyện và đánh giá Pipeline... ---")

# a. Định nghĩa các bước trong Pipeline
label_indexer = StringIndexer(inputCol="demand_level", outputCol="label", handleInvalid="keep")
day_type_indexer = StringIndexer(inputCol="day_type", outputCol="day_type_index", handleInvalid="keep")
day_type_encoder = OneHotEncoder(inputCol="day_type_index", outputCol="day_type_vec")
feature_columns = ["PULocationID", "hour", "day_of_week", "day_of_month", "month", "lag_24hr", "day_type_vec","lag_168hr"]
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features", handleInvalid="skip")
rf_classifier = RandomForestClassifier(featuresCol="features", labelCol="label", weightCol="weight", seed=42)
pipeline = Pipeline(stages=[label_indexer, day_type_indexer, day_type_encoder, assembler, rf_classifier])

# b. Chia dữ liệu Train/Test
train_data, test_data = final_data_for_pipeline.randomSplit([0.8, 0.2], seed=42)

# c. Huấn luyện Pipeline
pipeline_model = pipeline.fit(train_data)
print("Đã huấn luyện xong Pipeline!")

# d. Thực hiện dự đoán
predictions = pipeline_model.transform(test_data)

# e. Đánh giá mô hình
print("\n--- Bảng kết quả đánh giá mô hình ---")
evaluator_accuracy = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
accuracy = evaluator_accuracy.evaluate(predictions)
f1_score = evaluator_f1.evaluate(predictions)
print(f"Độ chính xác (Accuracy): {accuracy * 100:.2f}%")
print(f"Chỉ số F1-Score (trung bình): {f1_score:.2f}")

# f. Hiển thị kết quả chi tiết
print("\n--- Ma trận Nhầm lẫn (Confusion Matrix) ---")
labels = pipeline_model.stages[0].labels
prediction_converter = IndexToString(inputCol="prediction", outputCol="predicted_level", labels=labels)
final_output = prediction_converter.transform(predictions)
final_output.groupBy("demand_level").pivot("predicted_level", labels).count().na.fill(0).show()


--- Cách 2_BƯỚC 4: Bắt đầu xây dựng, huấn luyện và đánh giá Pipeline... ---
Đã huấn luyện xong Pipeline!

--- Bảng kết quả đánh giá mô hình ---
Độ chính xác (Accuracy): 65.69%
Chỉ số F1-Score (trung bình): 0.65

--- Ma trận Nhầm lẫn (Confusion Matrix) ---
+------------+--------+------+-----+-----+---------+
|demand_level|Very_Low|Medium|  Low| High|Very_High|
+------------+--------+------+-----+-----+---------+
|        High|      77|  2738|   76|25632|     5533|
|         Low|   20094|  4960|13036|  126|       50|
|   Very_High|      12|    44|    4|  699|     8626|
|      Medium|    1963| 26703| 5171| 5825|      255|
|    Very_Low|   39719|  1687| 8478|   68|       32|
+------------+--------+------+-----+-----+---------+



In [15]:
from pyspark.sql.functions import min, max, col

print("--- Phân tích khoảng giá trị (số lượng chuyến đi) cho mỗi mức nhu cầu ---")

# Sử dụng DataFrame đã có cột demand_level
# Giả sử DataFrame đó là final_data_for_pipeline
demand_ranges = final_data_for_pipeline.groupBy("demand_level") \
    .agg(
        min("trip_count").alias("min_trip_count"),
        max("trip_count").alias("max_trip_count")
    ) \
    .orderBy(col("min_trip_count"))

# Hiển thị kết quả
demand_ranges.show()

--- Phân tích khoảng giá trị (số lượng chuyến đi) cho mỗi mức nhu cầu ---
+------------+--------------+--------------+
|demand_level|min_trip_count|max_trip_count|
+------------+--------------+--------------+
|    Very_Low|             1|             1|
|         Low|             2|             5|
|      Medium|             6|            48|
|        High|            49|           185|
|   Very_High|           186|           991|
+------------+--------------+--------------+



In [16]:
# Giả sử bạn đã có mô hình tốt nhất là 'final_model' từ các bước trước

print("--- Bắt đầu quá trình dự đoán cho dữ liệu mới ---")

# --- Bước 1: Định nghĩa dữ liệu mới cần dự đoán ---
# Ví dụ: chúng ta muốn dự đoán nhu cầu cho:
# - Địa điểm (PULocationID): 132 (Sân bay JFK)
# - Thời gian: 18 giờ (6 giờ tối), thứ Sáu, ngày 10 tháng 5 năm 2024
# - Giả sử đây là một ngày bình thường (Normal_Day)
# - Giả sử nhu cầu giờ này hôm qua (lag_24hr) là 150 chuyến
# - Giả sử nhu cầu giờ này tuần trước (lag_168hr) là 180 chuyến

new_data_to_predict = [
    (132, 18, 6, 10, 5, 150.0, 180.0, "Normal_Day") 
    # PULocationID, hour, day_of_week, day_of_month, month, lag_24hr, lag_168hr, day_type
]

# --- Bước 2: Tạo Schema cho DataFrame mới ---
# Schema phải khớp chính xác với tên cột và kiểu dữ liệu
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType

schema = StructType([
    StructField("PULocationID", IntegerType(), True),
    StructField("hour", IntegerType(), True),
    StructField("day_of_week", IntegerType(), True),
    StructField("day_of_month", IntegerType(), True),
    StructField("month", IntegerType(), True),
    StructField("lag_24hr", DoubleType(), True),
    StructField("lag_168hr", DoubleType(), True),
    StructField("day_type", StringType(), True)
])

# --- Bước 3: Tạo DataFrame mới từ dữ liệu và schema ---
new_df = spark.createDataFrame(data=new_data_to_predict, schema=schema)

print("Dữ liệu đầu vào để dự đoán:")
new_df.show()
# --- Bước 4: Sử dụng mô hình đã huấn luyện để thực hiện dự đoán ---
# Dòng này sẽ tạo ra cột 'prediction' (dạng số)
raw_predictions = pipeline_model.transform(new_df)

# --- Bước 5: Thêm bước chuyển đổi từ số sang chữ ---
# Chúng ta cần lấy lại bộ chuyển đổi (StringIndexer) từ pipeline đã huấn luyện để biết nhãn nào tương ứng với số nào.
from pyspark.ml.feature import IndexToString

# Lấy ra các nhãn gốc (ví dụ: "High", "Low",...) từ giai đoạn đầu tiên của pipeline
label_converter = IndexToString(
    inputCol="prediction",
    outputCol="predicted_level",
    labels=pipeline_model.stages[0].labels # stages[0] chính là label_indexer
)

# Áp dụng bộ chuyển đổi để tạo ra cột 'predicted_level'
final_prediction_result = label_converter.transform(raw_predictions)


# --- Bước 6: Hiển thị kết quả dự đoán cuối cùng ---
print("Kết quả dự đoán nhu cầu:")
final_prediction_result.select(
    "PULocationID",
    "hour",
    "day_of_week",
    "predicted_level" # Bây giờ cột này đã tồn tại
).show()

--- Bắt đầu quá trình dự đoán cho dữ liệu mới ---
Dữ liệu đầu vào để dự đoán:
+------------+----+-----------+------------+-----+--------+---------+----------+
|PULocationID|hour|day_of_week|day_of_month|month|lag_24hr|lag_168hr|  day_type|
+------------+----+-----------+------------+-----+--------+---------+----------+
|         132|  18|          6|          10|    5|   150.0|    180.0|Normal_Day|
+------------+----+-----------+------------+-----+--------+---------+----------+

Kết quả dự đoán nhu cầu:
+------------+----+-----------+---------------+
|PULocationID|hour|day_of_week|predicted_level|
+------------+----+-----------+---------------+
|         132|  18|          6|      Very_High|
+------------+----+-----------+---------------+



In [17]:
from pyspark.sql.functions import lit, to_date, dayofweek, dayofmonth, month, col, broadcast
from pyspark.sql.types import IntegerType
from pyspark.ml.feature import IndexToString

print("--- BẮT ĐẦU DỰ ĐOÁN LẠI CHO NGÀY 1/1/2025 TRÊN TOÀN NYC ---")

# --- Bước 1: Tạo lưới dữ liệu (Grid) cho ngày dự đoán ---
# Lưới này bao gồm tất cả các địa điểm (PULocationID từ 1-263) và tất cả các giờ trong ngày (0-23).

# Tạo DataFrame chứa tất cả các giờ (0-23)
hours_df = spark.range(0, 24).withColumnRenamed("id", "hour")

# Tạo DataFrame chứa tất cả các địa điểm (1-263)
locations_df = spark.range(1, 264).withColumnRenamed("id", "PULocationID") \
    .withColumn("PULocationID", col("PULocationID").cast(IntegerType()))

# Sử dụng Cross Join để tạo mọi tổ hợp (địa điểm, giờ)
# Dùng broadcast() để tối ưu hóa join
prediction_grid = locations_df.crossJoin(broadcast(hours_df))

# --- Bước 2: Thêm các đặc trưng về thời gian và loại ngày ---
target_date = "2025-01-01"
prediction_grid = prediction_grid.withColumn("date", to_date(lit(target_date))) \
                                 .withColumn("day_of_week", dayofweek(col("date"))) \
                                 .withColumn("day_of_month", dayofmonth(col("date"))) \
                                 .withColumn("month", month(col("date"))) \
                                 .withColumn("day_type", lit("Holiday")) # 1/1 là ngày lễ (New Year's Day)

# --- Bước 3: Lấy và kết hợp dữ liệu quá khứ (Lag Features) ---
# Chúng ta cần nhu cầu của cùng giờ/địa điểm từ ngày hôm trước (2024-12-31) và tuần trước (2024-12-25).
# Dữ liệu này được lấy từ DataFrame `hourly_demand` đã tính toán ở bước [9].

# Dữ liệu lag 24 giờ (dữ liệu của ngày 31/12/2024)
lag_24hr_data = hourly_demand.filter(col("date") == "2024-12-31") \
                             .select(
                                 col("PULocationID"), 
                                 col("hour"), 
                                 col("trip_count").alias("lag_24hr")
                             )

# Dữ liệu lag 168 giờ (dữ liệu của ngày 25/12/2024 - Lễ Giáng Sinh)
lag_168hr_data = hourly_demand.filter(col("date") == "2024-12-25") \
                              .select(
                                  col("PULocationID"), 
                                  col("hour"), 
                                  col("trip_count").alias("lag_168hr")
                              )

# Kết hợp dữ liệu lag vào lưới dự đoán. Sử dụng left join và điền 0 cho các giá trị thiếu.
future_data_to_predict = prediction_grid.join(lag_24hr_data, ["PULocationID", "hour"], "left") \
                                          .join(lag_168hr_data, ["PULocationID", "hour"], "left") \
                                          .na.fill(0, subset=["lag_24hr", "lag_168hr"])

print(f"Đã tạo {future_data_to_predict.count()} điểm dữ liệu để dự đoán cho ngày {target_date}.")
print("Dữ liệu đầu vào mẫu:")
future_data_to_predict.show(5)

# --- Bước 4: Áp dụng Pipeline để dự đoán ---
# Sử dụng pipeline_model đã huấn luyện từ "Cách 2" để biến đổi dữ liệu mới và đưa ra dự đoán.
# Pipeline sẽ tự động xử lý việc chuyển đổi 'day_type' và lắp ráp các features.
future_predictions_raw = pipeline_model.transform(future_data_to_predict)


# --- Bước 5: Chuyển đổi nhãn dự đoán từ số sang chữ ---
# Sử dụng lại bộ chuyển đổi IndexToString đã được định nghĩa trong pipeline.
label_converter = IndexToString(
    inputCol="prediction",
    outputCol="predicted_level",
    labels=pipeline_model.stages[0].labels # Lấy nhãn từ bước đầu tiên (label_indexer) của pipeline
)
final_future_predictions = label_converter.transform(future_predictions_raw)


# --- Bước 6: Hiển thị kết quả dự đoán ---
print(f"\n--- KẾT QUẢ DỰ ĐOÁN NHU CẦU CHO NGÀY {target_date} ---")

# Xem ví dụ dự đoán cho Sân bay LaGuardia (PULocationID = 138)
print("\n>>> Dự đoán cho Sân bay LaGuardia (ID=138):")
final_future_predictions.filter(col("PULocationID") == 138) \
                        .select("hour", "predicted_level", "lag_24hr", "lag_168hr") \
                        .orderBy("hour") \
                        .show()

# Thống kê tổng quan về mức nhu cầu được dự đoán trên toàn thành phố
print("\n>>> Tổng quan dự đoán trên toàn thành phố:")
final_future_predictions.groupBy("predicted_level").count().orderBy(col("count").desc()).show()


# --- Bước 7: Xuất kết quả dự đoán ra file CSV ---
print("\n--- Đang xuất kết quả dự đoán ra file CSV ---")

# Chọn các cột cần thiết để xuất
output_df = final_future_predictions.select(
    "PULocationID", 
    "hour", 
    "date", 
    "predicted_level",
    "lag_24hr",
    "lag_168hr"
).orderBy("PULocationID", "hour")

# Ghi ra file CSV trên HDFS
# .coalesce(1) -> Gom dữ liệu vào 1 partition để xuất ra 1 file duy nhất.
# .option("header", "true") -> Thêm dòng tiêu đề cho các cột.
# .mode("overwrite") -> Nếu thư mục đã tồn tại, ghi đè lên nó.
output_path = "hdfs://master:9000/output/predictions_2025-01-01_rerun"
output_df.coalesce(1) \
         .write \
         .option("header", "true") \
         .mode("overwrite") \
         .csv(output_path)

print(f"Đã xuất file CSV thành công vào thư mục HDFS: {output_path}")

--- BẮT ĐẦU DỰ ĐOÁN LẠI CHO NGÀY 1/1/2025 TRÊN TOÀN NYC ---
Đã tạo 6312 điểm dữ liệu để dự đoán cho ngày 2025-01-01.
Dữ liệu đầu vào mẫu:
+------------+----+----------+-----------+------------+-----+--------+--------+---------+
|PULocationID|hour|      date|day_of_week|day_of_month|month|day_type|lag_24hr|lag_168hr|
+------------+----+----------+-----------+------------+-----+--------+--------+---------+
|           1|   0|2025-01-01|          4|           1|    1| Holiday|       0|        0|
|           1|   3|2025-01-01|          4|           1|    1| Holiday|       0|        0|
|           1|   5|2025-01-01|          4|           1|    1| Holiday|       0|        0|
|           1|   4|2025-01-01|          4|           1|    1| Holiday|       0|        0|
|           1|   1|2025-01-01|          4|           1|    1| Holiday|       0|        0|
+------------+----+----------+-----------+------------+-----+--------+--------+---------+
only showing top 5 rows


--- KẾT QUẢ DỰ ĐOÁN NHU CẦ